# Исходная GPT-2

Обучение на следующий ID трека.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "run.py").exists():
    raise RuntimeError("Открой ноутбук из корня проекта.")
RETRAIN = False  # True — обучение; False — сохранённые результаты.
WORK = ROOT / "runs/notebooks" if RETRAIN else ROOT

def run(stage):
    subprocess.run([sys.executable, str(ROOT / "run.py"), stage,
                    "--work-dir", str(WORK)], check=True)

## Подготовка и обучение

Временное разбиение; checkpoint по validation NDCG@10.

In [2]:
if RETRAIN:
    run("prepare")
    run("base")

## Выборки

Размеры и покрытие каталога из аудита данных.

In [3]:
audit_path = WORK / "data/processed/audit.json"
if audit_path.exists():
    audit = json.loads(audit_path.read_text())
else:
    audit = json.loads((WORK / "results/details.json").read_text())["data_audit"]
pd.DataFrame(audit["splits"]).T

,raw_events,catalog_target_fraction,examples,users,evaluation_catalog_fraction
train,4157149.0,0.416487,100000.0,969.0,1.000000
val,667739.0,0.362124,4618.0,930.0,0.368774
test,652216.0,0.323008,4927.0,1000.0,0.352141


В этом запуске NDCG@10: GPT-2 — 0.01851, популярность — 0.00487. Цели вне каталога учитываются как промахи.

In [4]:
metrics_path = WORK / "results/metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics[(metrics.split == "test") & metrics.model.isin(["popularity", "base"])][["model", "recall@10", "ndcg@10", "known_ndcg@10"]])
else:
    print("Обучение закончено. Общая оценка будет выполнена в третьем ноутбуке.")

,model,recall@10,ndcg@10,known_ndcg@10
24,popularity,0.01030,0.004874,0.013890
25,base,0.03045,0.018507,0.054091
